In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("./data_processed.csv")

In [3]:
data.head()

,Product Name,Category,Dosage Form,Price,Trademark,Brand Origin,Country,Rating,Continent,General_function
0,"Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...",Chăm sóc cơ thể,Gel,105000.0,DECUMAR,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
1,Dung dịch vệ sinh vùng kín Bimunica 250ml dành...,Chăm sóc cơ thể,Dạng kem,230000.0,Eucerin,Mỹ,Liên Bang Nga,5.0,Europe,Chăm sóc cơ thể
2,"Kem giảm thâm vùng nách, mông, bikini Neothera...",Chăm sóc cơ thể,Dạng kem,139000.0,La Beauty,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
3,Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...,Chăm sóc cơ thể,Dạng kem,390000.0,SVR,Pháp,Pháp,unknown,Europe,Chăm sóc cơ thể
4,Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...,"Lăn khử mùi, xịt khử mùi",Dạng bọt,96000.0,Eucerin,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể


In [4]:
data.shape

(1999, 10)

In [5]:
# Create a label encoder object
label_encoder = preprocessing.LabelEncoder()

# Encode labels in the 'Country' column
data['Country'] = label_encoder.fit_transform(data['Country'])
data['Trademark'] = label_encoder.fit_transform(data['Trademark'])
data['General_function'] = label_encoder.fit_transform(data['General_function'])

print(data.head())

                                        Product Name  \
0  Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1  Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2  Kem giảm thâm vùng nách, mông, bikini Neothera...   
3  Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4  Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   

                   Category Dosage Form     Price  Trademark Brand Origin  \
0           Chăm sóc cơ thể         Gel  105000.0         75     Việt Nam   
1           Chăm sóc cơ thể    Dạng kem  230000.0        118           Mỹ   
2           Chăm sóc cơ thể    Dạng kem  139000.0        217     Việt Nam   
3           Chăm sóc cơ thể    Dạng kem  390000.0        375         Pháp   
4  Lăn khử mùi, xịt khử mùi    Dạng bọt   96000.0        118     Việt Nam   

   Country   Rating Continent  General_function  
0       37      5.0      Asia                 0  
1       15      5.0    Europe                 0  
2       37      5.0      Asia                 0  


In [6]:
train_data = data[data['Rating'] != 'unknown']  # Sản phẩm có rating
prediction_data = data[data['Rating'] == 'unknown']  # Sản phẩm chưa có rating

In [7]:
train_data.shape, prediction_data.shape

((1220, 10), (779, 10))

In [8]:
train_data.to_csv("D:\\FILE_CSV_DATA_MODELING\\train_data.csv")

### 2. Feature Selection

In [9]:
features = ['Price', 'Trademark', 'Country', 'General_function']

### 3. Splitting dataset into X and y

In [10]:
X = train_data[features]
y = train_data["Rating"]
X_test = prediction_data[features]
y_test = prediction_data["Rating"]

In [11]:
X.head()

,Price,Trademark,Country,General_function
0,105000.0,75,37,0
1,230000.0,118,15,0
2,139000.0,217,37,0
4,96000.0,118,37,0
5,132000.0,118,10,1


In [12]:
y.head()

0    5.0
1    5.0
2    5.0
4    5.0
5    5.0
Name: Rating, dtype: object

##### X,y -> X_train, y_train, X_valid, y_valid

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state=0)

In [14]:
X.shape, X_train.shape, X_valid.shape

((1220, 4), (976, 4), (244, 4))

### Model training

- Random Forest
- Gradient Boosting (XGBoost)
- Ridge Regression
- SVR(kernel=rbf)

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,  make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge

### **Random Forest algorithm**

In [16]:
#YOUR CODE HERE

### **Gradient Boosting algorithm**

In [17]:
#YOUR CODE HERE

### **Ridge Regression algorithm**

In [18]:
#YOUR CODE HERE

### **SVR(kernel=rbf)**

In [19]:
#YOUR CODE HERE

### Cross-validation (baseline model)

In [20]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
seed = 2024
models = [
    RandomForestRegressor(n_estimators=200, max_depth=10, random_state=seed),
    Ridge(alpha=1.0),
    #XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=seed)
]

rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

def generate_baseline_results(models, X, y, metrics, cv=5):
    #define k-fold:
    kflod = KFold(n_splits=cv, shuffle=True, random_state=seed)
    entries = []
    for model in models:
        model_name = model.__class__.__name__
        scores = cross_val_score(model, X, y, scoring=metrics, cv=kflod)
        for fold_idx, score in enumerate(scores):
            entries.append((model_name, fold_idx, -score))
    
    cv_df = pd.DataFrame(entries, columns=['model_name', 'fold_id', 'RMSE'])
    
    # Summary result
    mean = cv_df.groupby('model_name')['RMSE'].mean()

    baseline_results = pd.DataFrame(mean)
    baseline_results.columns = ['Mean']
    
    # Sort by RMSE 
    baseline_results.sort_values(by='Mean', ascending=True, inplace=True)
    
    return baseline_results

generate_baseline_results(models, X, y, metrics=rmse_scorer, cv=5)

,Mean
model_name,
Ridge,0.341859
RandomForestRegressor,0.387965
